# 05 — Explainable AI (XAI): LLM-Generated Clinical Reasoning

**Project:** End-to-End Heart Disease Prediction System
**Stage:** 5 / 5 — Explainable AI Reasoning

## Objective
A bare probability score ("87% risk of heart disease") is not actionable for a
clinician. This notebook closes that gap with a small, **locally-run** LLM
(`Qwen2.5-0.5B-Instruct`) that translates a prediction — together with the
specific clinical markers that triggered it — into a short, structured
narrative a medical reviewer can sanity-check at a glance.

**Why a local, sub-1B model instead of a hosted frontier LLM?**
- Patient-level clinical data should not need to leave a hospital's
  infrastructure to generate an explanation — a small model runs fully
  on-premises (CPU or a single consumer GPU).
- The job here is narrow (templated narrative generation from already-known
  structured facts), not open-ended reasoning, so a 0.5B model is sufficient
  and dramatically cheaper to host than a frontier API.

**Design pattern:** the LLM is *not* asked to discover risk factors on its
own — that would invite hallucination. A deterministic, rule-based function
first extracts the clinically meaningful flags from the structured data; the
LLM's only job is to phrase those already-verified facts as fluent prose.
This keeps the diagnostic logic auditable while still gaining natural-language
fluency.

> **Important limitation to state explicitly (and worth saying out loud in an
> interview):** this is a decision-support narrative generator, not a
> diagnostic tool. Outputs must be reviewed by a qualified clinician before
> any action is taken, and the LLM should never be the source of the risk
> determination itself — only its narrator.

## Input
- `models/final_ensemble.pkl` (Notebook 04)
- `data/processed/test_clean.csv` (Notebook 03) — for model inference
- `data/interim/test_imputed.csv` (Notebook 01) — for human-readable clinical
  fields (chest pain type, ST slope, etc., which no longer exist in their
  original form after one-hot encoding)

## Output
- A printed / saved clinical reasoning report for the highest-confidence
  positive predictions

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROCESSED_DIR = "../data/processed"
INTERIM_DIR   = "../data/interim"
MODELS_DIR    = "../models"
REPORTS_DIR   = "../reports"

TOP_N = 5  # number of highest-confidence positive cases to explain

## 2. Load Final Ensemble & Test Data

In [2]:
final_ensemble = joblib.load(f"{MODELS_DIR}/final_ensemble.pkl")

test_clean   = pd.read_csv(f"{PROCESSED_DIR}/test_clean.csv")
test_raw_fields = pd.read_csv(f"{INTERIM_DIR}/test_imputed.csv")  # still has original categorical values

y_test_proba = final_ensemble.predict_proba(test_clean)[:, 1]
y_test_pred  = (y_test_proba >= 0.5).astype(int)  # use model_leaderboard.pkl / model_clinical_safety.pkl threshold in production

results = test_raw_fields.copy()
results['HeartDisease_pred'] = y_test_pred
results['Confidence'] = (y_test_proba * 100).round(1)

print(f"Test set scored: {len(results)} patients")
print(f"Predicted Disease (1): {(y_test_pred == 1).sum()}")

Test set scored: 184 patients
Predicted Disease (1): 106


## 3. Rule-Based Risk Factor Extraction

Every "risk factor" string below is generated **deterministically from the
structured data** — the LLM never invents these; it only narrates them. This
function is the auditable core of the explanation: a clinician (or a unit
test) can verify every flag against the raw patient values independently of
the LLM output.

In [3]:
def extract_risk_factors(row):
    risk_factors = []

    if row.get('ChestPainType') == 'ASY':
        risk_factors.append(
            "ChestPainType = ASY (Asymptomatic): the absence of chest pain is "
            "itself a recognized red flag, since silent ischemia is frequently "
            "missed in clinical practice precisely because it is painless."
        )
    if float(row.get('Oldpeak', 0)) > 1.5:
        risk_factors.append(
            f"Oldpeak = {row.get('Oldpeak')} mm (elevated): significant ST-segment "
            "depression indicates myocardial ischemia during physical exertion."
        )
    if row.get('ST_Slope') in ['Flat', 'Down']:
        risk_factors.append(
            f"ST_Slope = {row.get('ST_Slope')}: an abnormal ST-segment slope during "
            "the exercise stress test suggests impaired myocardial perfusion."
        )
    if row.get('ExerciseAngina') == 'Y':
        risk_factors.append(
            "ExerciseAngina = Y: chest pain provoked by exercise is a strong "
            "clinical confirmation of activity-induced coronary ischemia."
        )
    if float(row.get('MaxHR', 999)) < 120:
        risk_factors.append(
            f"MaxHR = {row.get('MaxHR')} bpm: chronotropic capacity is markedly "
            f"limited for a patient aged {row.get('Age')}."
        )
    if float(row.get('RestingBP', 0)) > 140:
        risk_factors.append(
            f"RestingBP = {row.get('RestingBP')} mmHg: Stage 2 hypertension chronically "
            "increases left-ventricular workload."
        )
    if not risk_factors:
        risk_factors.append(
            f"The combination of age {row.get('Age')} with the broader clinical profile "
            f"(RestingBP={row.get('RestingBP')}, Cholesterol={row.get('Cholesterol')}) "
            "reflects an accumulation of cardiovascular risk factors rather than any "
            "single dominant marker."
        )
    return risk_factors

## 4. Load the Local LLM

In [4]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print(f"Loaded {MODEL_ID} on device: {llm_model.device}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


Loaded Qwen/Qwen2.5-0.5B-Instruct on device: cpu


## 5. Prompt Template

The prompt constrains the LLM to a fixed three-sentence structure, each with
a mandated opening clause. This is a lightweight but effective guardrail
against rambling or off-topic generations from a small model.

In [5]:
SYSTEM_PROMPT = (
    "You are a senior cardiologist explaining a machine learning model's "
    "prediction to a clinical team. Provide an analysis that is accurate, "
    "evidence-based, and easy to understand."
)

def build_prompt(row, risk_factors):
    main_feature = risk_factors[0].split(':')[0]
    return (
        "Write exactly 3 formal clinical sentences in English.\n\n"
        f"Patient data: age {row.get('Age')}, sex {'male' if row.get('Sex') == 'M' else 'female'}, "
        f"ChestPainType={row.get('ChestPainType')}, Oldpeak={row.get('Oldpeak')}, "
        f"ST_Slope={row.get('ST_Slope')}, ExerciseAngina={row.get('ExerciseAngina')}.\n\n"
        f"Sentence 1 (must start with 'This patient'): explain the clinical "
        f"significance of {main_feature}.\n"
        f"Sentence 2 (must start with 'The Oldpeak value'): explain the implication "
        f"of Oldpeak={row.get('Oldpeak')} together with ST_Slope={row.get('ST_Slope')}.\n"
        "Sentence 3 (must start with 'The combination of these factors'): conclude "
        "why this patient is at elevated risk of heart disease.\n\n"
        "Output only the 3 sentences, no numbering, no preamble."
    )


def generate_reasoning(prompt, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        generated = llm_model.generate(
            inputs.input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = [g[len(i):] for i, g in zip(inputs.input_ids, generated)]
    return tokenizer.batch_decode(generated, skip_special_tokens=True)[0].strip()

## 6. Generate Reasoning for the Top-N Highest-Confidence Positive Cases

`TOP_N` is set in the configuration cell above (default: 5).

In [6]:
sample_patients = (
    results[results['HeartDisease_pred'] == 1]
    .nlargest(TOP_N, 'Confidence')
    .reset_index(drop=True)
)

report_lines = []

for _, row in sample_patients.iterrows():
    risk_factors = extract_risk_factors(row)
    prompt = build_prompt(row, risk_factors)
    reasoning = generate_reasoning(prompt)

    bullets = "\n".join([f"  - {rf}" for rf in risk_factors])
    block = (
        f"\n{'='*70}\n"
        f"Patient ID        : #{row.get('id')}\n"
        f"Model Prediction  : HEART DISEASE (1)\n"
        f"Confidence Score  : {row.get('Confidence')}%\n\n"
        f"Rule-Based Risk Factors Detected:\n{bullets}\n\n"
        f"LLM Clinical Reasoning ({MODEL_ID}):\n{reasoning}\n"
        f"{'='*70}"
    )
    report_lines.append(block)
    print(block)

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Patient ID        : #443
Model Prediction  : HEART DISEASE (1)
Confidence Score  : 98.2%

Rule-Based Risk Factors Detected:
  - ChestPainType = ASY (Asymptomatic): the absence of chest pain is itself a recognized red flag, since silent ischemia is frequently missed in clinical practice precisely because it is painless.
  - ST_Slope = Flat: an abnormal ST-segment slope during the exercise stress test suggests impaired myocardial perfusion.
  - ExerciseAngina = Y: chest pain provoked by exercise is a strong clinical confirmation of activity-induced coronary ischemia.

LLM Clinical Reasoning (Qwen/Qwen2.5-0.5B-Instruct):
This patient has Asymptomatic Chest Pain Type (ASY). The Oldpeak value of 1.2 indicates mild exertion may trigger angina. The ST_Slope being Flat suggests stable coronary artery disease. These factors increase the risk of heart disease.



Patient ID        : #351
Model Prediction  : HEART DISEASE (1)
Confidence Score  : 97.9%

Rule-Based Risk Factors Detected:
  - ChestPainType = ASY (Asymptomatic): the absence of chest pain is itself a recognized red flag, since silent ischemia is frequently missed in clinical practice precisely because it is painless.
  - ST_Slope = Flat: an abnormal ST-segment slope during the exercise stress test suggests impaired myocardial perfusion.

LLM Clinical Reasoning (Qwen/Qwen2.5-0.5B-Instruct):
This patient has Asymptomatic Chest Pain Type (Chest pain without typical symptoms). The Oldpeak value of 0.0 suggests minimal exertion does not exacerbate their condition. The ST_Slope being Flat indicates stable angina, which is a common sign of coronary artery disease. Together, these factors increase the risk of developing heart disease.



Patient ID        : #532
Model Prediction  : HEART DISEASE (1)
Confidence Score  : 97.5%

Rule-Based Risk Factors Detected:
  - ChestPainType = ASY (Asymptomatic): the absence of chest pain is itself a recognized red flag, since silent ischemia is frequently missed in clinical practice precisely because it is painless.
  - Oldpeak = 1.8 mm (elevated): significant ST-segment depression indicates myocardial ischemia during physical exertion.
  - ST_Slope = Flat: an abnormal ST-segment slope during the exercise stress test suggests impaired myocardial perfusion.
  - ExerciseAngina = Y: chest pain provoked by exercise is a strong clinical confirmation of activity-induced coronary ischemia.
  - MaxHR = 115 bpm: chronotropic capacity is markedly limited for a patient aged 64.
  - RestingBP = 143 mmHg: Stage 2 hypertension chronically increases left-ventricular workload.

LLM Clinical Reasoning (Qwen/Qwen2.5-0.5B-Instruct):
This patient has Asymptomatic Chest Pain Type (ChestPainType = ASY),


Patient ID        : #413
Model Prediction  : HEART DISEASE (1)
Confidence Score  : 97.2%

Rule-Based Risk Factors Detected:
  - ChestPainType = ASY (Asymptomatic): the absence of chest pain is itself a recognized red flag, since silent ischemia is frequently missed in clinical practice precisely because it is painless.
  - ST_Slope = Flat: an abnormal ST-segment slope during the exercise stress test suggests impaired myocardial perfusion.
  - ExerciseAngina = Y: chest pain provoked by exercise is a strong clinical confirmation of activity-induced coronary ischemia.
  - MaxHR = 103 bpm: chronotropic capacity is markedly limited for a patient aged 56.

LLM Clinical Reasoning (Qwen/Qwen2.5-0.5B-Instruct):
This patient has Asymptomatic Chest Pain Type (ASY), indicating no chest pain symptoms. The Oldpeak value of 1.0 suggests minimal exertion or rest can cause angina, which is common in patients with ASY. Additionally, the ST_Slope being Flat indicates that there is no significant elevati


Patient ID        : #585
Model Prediction  : HEART DISEASE (1)
Confidence Score  : 97.1%

Rule-Based Risk Factors Detected:
  - ChestPainType = ASY (Asymptomatic): the absence of chest pain is itself a recognized red flag, since silent ischemia is frequently missed in clinical practice precisely because it is painless.
  - ST_Slope = Flat: an abnormal ST-segment slope during the exercise stress test suggests impaired myocardial perfusion.
  - ExerciseAngina = Y: chest pain provoked by exercise is a strong clinical confirmation of activity-induced coronary ischemia.
  - MaxHR = 116 bpm: chronotropic capacity is markedly limited for a patient aged 64.
  - RestingBP = 141 mmHg: Stage 2 hypertension chronically increases left-ventricular workload.

LLM Clinical Reasoning (Qwen/Qwen2.5-0.5B-Instruct):
This patient has Asymptomatic Chest Pain Type (ChestPainType = ASY), indicating mild or no chest pain. The Oldpeak value of 1.5 suggests minimal exertion during exercise, which could be indic

## 7. Persist the Clinical Reasoning Report

In [7]:
report_path = f"{REPORTS_DIR}/clinical_reasoning_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print(f"\n[DONE] Clinical reasoning report saved to: {report_path}")


[DONE] Clinical reasoning report saved to: ../reports/clinical_reasoning_report.txt


---
## Closing Notes & Limitations

- **This is a decision-support artifact, not a diagnosis.** Every output here
  must be reviewed by a qualified clinician before influencing patient care.
- **Risk factors are rule-based and fully auditable**; only their phrasing is
  LLM-generated, which bounds the model's ability to hallucinate clinical facts.
- **A 0.5B local model is intentionally chosen** over a larger hosted model —
  for a templated narration task at this scope, model size is a deployment
  cost/privacy trade-off, not a quality bottleneck.
- **Possible extensions:** swap in a feature-attribution method (SHAP/LIME) to
  feed the LLM model-derived importances rather than hand-written clinical
  rules, or fine-tune the small LLM on a curated set of clinician-written
  example explanations for tighter style control.

This concludes the 5-notebook pipeline: **Data Cleaning → EDA → Feature
Engineering → Modeling & Ensemble → Explainable AI Reasoning.**